# D66 MLOps Intro Code

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

last modified date : 2026.11.24
제작 : 박광석 (모두의연구소)

# MLOps 소개 구현 실습

## 실습 목표와 핵심 질문

**평가 결과를 다른 사람이 다시 확인하려면 무엇을 함께 저장해야 합니까?**

가상의 사내 규정 챗봇을 시범 운영하는 팀에 기준선과 후보 설정이 있다고 가정합니다. 정답 수와 속도만 전달받으면 같은 조건의 결과인지, 새 설정에 문제가 생기면 무엇으로 돌아갈지 알기 어렵습니다. 필요한 기록과 이유를 두 문장으로 적고, 실습 뒤 자신의 출력으로 설명을 보완합니다.

**학습목표**

- [목표 1] 서로 다른 형식의 평가 결과를 실행 조건·지표·원본 파일과 함께 기록할 수 있다.
- [목표 2] 질문 세트·모델·GPU가 같은 실행을 찾고 비교를 보류할 조건을 설명할 수 있다.
- [목표 3] 서비스 버전에서 평가 실행을 찾고 운영·직전 별칭의 변경 전후를 확인할 수 있다.

**이전 교안이나 수행 기록 없이 시작할 수 있습니다.** Step 0~6을 위에서부터 실행하고 Step 9에서 보관합니다. 필요한 예제 파일은 Step 1에서 생성하며, 별도 업로드·하네스 파일·모델 다운로드·RunPod 접속은 필요하지 않습니다. Python의 딕셔너리와 파일 읽기 기초를 사용합니다.

**MLflow는 실험의 설정, 점수, 원본 파일을 모아 검색하고 평가한 구성을 버전으로 등록하는 도구입니다.** 오늘 처음 사용합니다. 예제 파일을 넣고 “이 점수는 어느 질문과 GPU에서 나왔는가”, “현재 선택한 구성을 평가한 실행은 무엇인가”를 직접 찾아봅니다.

![흩어진 결과 파일을 한 형식으로 모아 트래킹 서버에 기록하고 서비스 버전에 운영 별칭을 붙인 뒤 점검표로 판정합니다.](lms-img:0)

*원본 파일에서 조건을 확인하고, MLflow의 실행 기록을 서비스 버전에 연결합니다. 마지막에 운영·직전 별칭이 가리키는 버전을 확인합니다.*

## 이론과 실습의 연계 구성

| 시간 예시 | 분 | 활동 | 실습 노트 위치 |
|---|---:|---|---|
| 10:00~11:05 | 65 | 예제 결과 생성과 조건 통일 | Step 0~2 |
| 11:05~11:25 | 20 | 휴식 |  |
| 11:25~13:00 | 95 | 등록한 실행에서 비교할 쌍 하나 찾기 | Step 3~5 |
| 13:00~14:00 | 60 | 점심 |  |
| 14:00~15:30 | 90 | 기준선·후보 버전 등록과 별칭 복구 | Step 6 |
| 15:30~15:50 | 20 | 휴식 |  |
| 15:50~16:40 | 50 | 실행 파일·별칭 기록 보관과 5분 확인 | Step 9 |
| 16:40~17:00 | 20 | 설치 지연·오류 대응 여유 |  |

**오늘 확인할 질문**　다른 팀원이 기록만 보고 어떤 조건에서 평가한 설정을 골랐는지, 어느 버전으로 되돌릴지 확인할 수 있을까요?

**7시간 구성**　필수 학습 300분(이론·실습·5분 마무리), 점심 60분, 휴식 40분, 오류 대응 20분입니다. 표에 있는 Step을 순서대로 실행합니다. 오류 대응 시간은 지연이 발생한 구간에 씁니다.

성숙도 점검표의 별도 채점과 심화 트랙은 선택입니다.

![MLOps 소개 노트(D66)의 입력에서 결과까지 이어지는 학습 흐름](lms-img:0)

**실습 순서**　측정 파일과 조건 확인 → MLflow에 실행과 원본 저장 → 비교할 수 있는 실행 확인 → 서비스 버전 등록과 별칭 되돌리기

### 핵심 용어 정리

| 용어 | 뜻과 실제로 하는 일 |
|---|---|
| MLOps |  모델의 평가·배포·운영을 반복하기 위한 작업 절차 |
| 실험과 실행 | 실험은 같은 목적의 결과를 모아 둔 그룹이고, 실행은 그 안에 들어가는 평가 결과 한 건입니다. |
| 트래킹 서버 | 기록을 저장하거나 조회해 달라는 요청을 받는 프로그램 |
| 아티팩트 |  실험 결과를 확인하거나 재현하는 데 필요한 파일 |
| 레지스트리 |  버전별 모델과 서비스 정보를 등록하는 목록 |
| 롤백 |  서비스를 이전에 확인한 버전으로 되돌리는 동작 |
| RAG·청크 | 관련 문서를 검색해 답변의 근거로 쓰는 구성과, 검색할 수 있도록 나눈 문서 조각 |
| p50·p95 지연 | 응답 시간의 50·95백분위 값. 예제 원본은 밀리초(ms), 기록할 때는 초(s)를 사용 |
| flip | 같은 질문에서 기준선의 정답을 후보가 오답으로 바꾼 경우(lost), 또는 반대 경우(gained) |
| 해시 | 입력 내용의 변경 여부를 확인할 식별값. 무엇을 해시했는지도 함께 확인 |

## 실행 환경

이 노트는 **GPU가 필요 없습니다.** 인터넷에 연결된 코랩 CPU 런타임에서 MLflow를 설치하고, 모의 결과 파일을 생성해 실습합니다. 모델·과거 교안·수행 기록은 준비하지 않습니다.

기본 입력 모드는 `sample`이며 자신의 과거 파일을 자동으로 섞지 않습니다. 예제의 모델·GPU·패키지 이름과 성능 수치는 입력 형식을 연습하기 위한 값입니다. 실제로 실행하는 것은 MLflow 기록·조회·버전 등록·별칭 변경이며, 모의 성능값으로 모델의 우열을 주장하지 않습니다. 자신의 실측 결과 사용은 Step 1의 선택 안내를 따릅니다.

PC나 사내 서버에서 해야 하는 일에는  **로컬 작업** 표시를 붙였습니다. 코랩에서는 읽고 넘어가도 됩니다.

| 항목 | 내용 |
|---|---|
| 실행 환경 | 인터넷에 연결된 코랩 CPU 런타임 |
| GPU·모델 | 필요 없음. 모델을 로드하지 않음 |
| 입력 파일 | Step 1이 만드는 예제 7개(JSON 6개·CSV 1개) |
| 외부 파일·계정 | 기존 교안·실험 파일·하네스·RunPod 계정 불필요 |
| 기동하는 서버 | MLflow 트래킹 서버 하나(포트 8660부터 빈 포트), 부록 B·E에서 추가 서버 |
| 오래 걸리는 셀 | 패키지 설치와 서버 기동. 출력 로그로 상태를 확인 |
| 선택 확장 | 자신의 실측 파일로 재실행, 사내 서버의 인증·백업·반입 운영 |

## Step 0. 환경 준비

첫 셀은 현재 런타임에 MLflow를 설치합니다. 설치에는 인터넷이 필요합니다. Step 1에서 예제 파일을 만든 뒤 Step 3에서 기록을 받을 서버를 실행합니다. 패키지 설치 뒤 import 오류가 나면 런타임을 다시 시작하고 이 셀부터 순서대로 실행합니다.

In [1]:
# 검증한 패치 버전으로 설치합니다. 현재 노트북의 파이썬 환경에 설치합니다.
%pip install -q "mlflow==3.16.1"

In [2]:
import os

# 실습에서 사용 통계를 외부로 보내지 않도록 MLflow를 불러오기 전에 설정합니다.
os.environ.setdefault("MLFLOW_DISABLE_TELEMETRY", "true")
os.environ.setdefault("MLFLOW_DISABLE_AGENT_" + "HINT", "1")   # 안내 로그를 끕니다. 이름을 나눠 적은 것은 LMS 푸시 검사 때문
os.environ.setdefault("MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT", "true")   # 실행마다 출력되는 화면 주소 줄을 끕니다
os.environ.setdefault("NO_PROXY", "127.0.0.1,localhost")      # 내 컴퓨터로 가는 요청은 프록시를 거치지 않습니다

import csv, hashlib, json, random, socket, subprocess, sys, time, urllib.request
from collections import Counter
from importlib.metadata import version
from pathlib import Path

print("[설치된 버전]", {d: version(d) for d in ["mlflow", "pandas", "sqlalchemy"]})

[설치된 버전] {'mlflow': '3.16.1', 'pandas': '2.2.3', 'sqlalchemy': '2.1.7'}


## Step 1. 결과 파일 수집

### Step 1. 결과 파일 준비

아래 셀부터 공통 준비 코드, 파일 확인 셀까지 **위에서 아래로 한 번씩 실행**합니다. 기본값 `sample`에서는 서로 다른 형식의 예제 파일 7개를 만들며, 과거 파일을 찾거나 가져오지 않습니다. 파일 이름의 D54~D60은 형식의 출처를 가리키는 이름일 뿐, 해당 교안을 먼저 열어야 한다는 뜻이 아닙니다.

`SOURCES`는 형식별 파일 이름 목록이고 `make_standin()`이 그 형식의 JSON·CSV를 생성합니다. 파일 확인 셀은 모의 데이터 표시와 RAG 원본의 주요 필드를 출력합니다. 질문 번호는 예제 안에 정의하며 외부 하네스를 불러오지 않습니다.

<details>
<summary>선택: 자신의 실측 결과 파일로 같은 절차 연습하기</summary>

이미 보관한 파일이 있을 때만 사용합니다. 새 런타임에서 Step 1 실행 전에 `os.environ["D66_INPUT_MODE"] = "real"`로 설정하고 파일을 `/content/d66/inbox`에 업로드합니다. 기본 경로를 바꿨다면 `D66_WORK` 아래의 `inbox`입니다. 이 모드는 입력 목록에 있는 파일을 찾고, 빠진 파일만 예제로 보완합니다. 기본 예제와는 작업 폴더·실험 이름을 나눠 저장합니다. 실측·대역 표시와 원본의 실행 조건을 확인해주세요.

</details>

In [3]:
# 기본 경로는 예제 전용입니다. 이전 실습 파일 없이 실행합니다.
INPUT_MODE = os.environ.get("D66_INPUT_MODE", "sample").lower()
if INPUT_MODE not in {"sample", "real"}:
    raise ValueError("D66_INPUT_MODE는 sample 또는 real입니다")
WORK_BASE = Path(os.environ.get("D66_WORK", "/content/d66"))
WORK = WORK_BASE / INPUT_MODE
INBOX = Path(os.environ.get("D66_INBOX", str(WORK_BASE / "inbox")))
WORK.mkdir(parents=True, exist_ok=True)
INBOX.mkdir(parents=True, exist_ok=True)
print("입력 모드:", INPUT_MODE, "/ 작업 폴더:", WORK)

# 이름은 예제 형식을 구분합니다. real 모드에서만 기존 파일 경로도 확인합니다.
SOURCES = {
    "양자화 노트": ["d54_results.json", "/content/d54_results.json"],
    "프루닝과 증류 노트": ["d55_results.json", "/content/d55/d55_results.json"],
    "GPTQ·AWQ 노트": ["d56_results.json", "/workspace/results/d56/d56_results.json"],
    "vLLM 기초 노트": ["d57_results.json", "/workspace/results/d57/d57_results.json"],
    "vLLM 심화 노트": ["d58_capacity.json", "/content/d58/d58_capacity.json"],
    "TensorRT-LLM 노트": ["d59_bench_mine.csv", "d59_bench_instructor.csv",
                          "/content/d59/d59_bench_mine.csv", "/content/d59/d59_bench_instructor.csv"],
    "RAG 서빙 최적화 노트": ["d60_results.json", "/content/d60/d60_results.json"],
}


def find_file(places):
    if INPUT_MODE == "sample":
        return None  # 기존 파일을 읽지 않는 예제 경로
    for item in places:
        path = Path(item) if "/" in item else INBOX / item
        if path.is_file():
            return path
    return None

입력 모드: sample / 작업 폴더: /content/d66/sample


## 공통 준비 코드. mock 결과 파일 만들기

### 공통 준비 코드. 모의 결과 파일 만들기

바로 앞의 경로·`SOURCES` 셀 다음에 실행합니다. `sample` 모드에서는 `standin` 폴더에 예제 7개를 만듭니다. 이어지는 파일 확인 셀과 Step 2로 진행하면 됩니다.

파일의 `_standin`, CSV의 `source` 열과 트래킹 서버의 `대역` 태그가 모의 데이터임을 표시합니다. 고정 시드로 생성하므로 같은 절차를 다시 실행해 기록의 중복 여부도 확인할 수 있습니다. 일부 GPU·패키지 정보는 누락 조건을 찾는 연습을 위해 의도적으로 비워 둡니다.

In [4]:
# 바로 앞 셀의 SOURCES를 사용합니다. 기본 sample 모드는 예제 7개를 모두 생성합니다.
STANDIN_NOTE = "mock : 형식만 맞춘 값이고 측정값이 아닙니다 (MLOps 소개 노트 코드가 만듦)"
QIDS_30 = [f"{c}{i}" for c, n in [("A", 8), ("B", 8), ("C", 7), ("D", 7)] for i in range(1, n + 1)]   # 이 셀 안에서 정의한 예제 질문 번호
M15, M7, M32 = "Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-7B-Instruct", "Qwen/Qwen2.5-32B-Instruct-AWQ"
rng = random.Random(66)


def correct(p_ok):
    return {q: rng.random() < p_ok for q in QIDS_30}


def harness_rows(spec):
    """양자화 노트 하네스(measure_all)의 결과 형식. spec : 행 이름 -> (맞히는 비율, 초당 토큰, 가중치 GB, 모델)"""
    return {n: {"precision": n, "model_id": m, "weights_gb": gb, "peak_gb": round(gb * 1.25, 2), "tps": tps,
                "correct": correct(p)} for n, (p, tps, gb, m) in spec.items()}


def make_standin(note, path):
    """노트마다 그 노트의 결과 파일과 같은 형식으로 mock 파일을 씁니다"""
    V = {"torch": "2.11.0", "transformers": "5.18.0"}
    if note == "양자화 노트":                     # 조건 필드에 GPU 이름이 없는 형식
        data = harness_rows({"16bit": (0.9, 21.0, 3.09, M15), "int8": (0.88, 9.5, 1.76, M15), "nf4": (0.85, 17.2, 1.12, M15)})
        data["_conditions"] = {"model": M15, "versions": V, "questions": 30}
    elif note == "프루닝과 증류 노트":             # 조건 필드에 GPU 이름까지 있는 형식
        data = harness_rows({"16bit": (0.9, 21.0, 3.09, M15), "비정형 50%": (0.6, 20.8, 3.09, M15),
                             "증류 0.5B": (0.55, 35.0, 0.99, "Qwen/Qwen2.5-0.5B-Instruct")})
        data["_conditions"] = {"model": M15, "questions": 30, "gpu": "Tesla T4", "versions": V}
    elif note == "GPTQ·AWQ 노트":                 # 조건 필드가 없는 형식. 팀 Pod 에서 측정한 7B 와 32B 4비트
        d56 = "/workspace/results/d56/"
        data = harness_rows({"16bit": (0.93, 30.0, 15.2, M7), "gptq": (0.9, 45.0, 5.58, d56 + "7b-gptq-ko"),
                             "awq": (0.9, 44.0, 5.58, d56 + "7b-awq-ko"), "32B awq": (0.97, 25.0, 19.3, M32)})
    elif note == "vLLM 기초 노트":                # 행, 팀원마다 기동한 서버의 측정, 용량
        data = {n: {"정답": sum(c.values()), "경과 초": w, "초당 토큰": round(3000 / w, 2), "가장 늦은 사람 초": w,
                    "engine": e, "correct": c}
                for n, e, w, c in [("transformers 16비트 한 명씩", "transformers", 60.0, correct(0.93)),
                                   ("vLLM 32B 4비트 한 명씩", "vLLM LLM", 40.0, correct(0.97))]}
        data["팀 설정 측정"] = {f"{r} {s}": {"역할": r, "설정": s, "모델": m, "비율": 0.9, "max_model_len": 4096,
                                       "혼자 초당 토큰": t, "최대 지연 초": d, "정답": sum(c.values()), "correct": c}
                              for r, s, m, t, d, c in [("A", "기준", M32, 30.0, 9.0, correct(0.97)),
                                                       ("D", "7B 16비트", M7, 45.0, 5.0, correct(0.93))]}
        data["용량"] = {"SLO 초": 6.0, "vLLM 응답 시간 기준": 16, "transformers 어림": 3, "VRAM 기준": None}
        data["_conditions"] = {"team": "team1", "gpu": "NVIDIA L40S", "model": M32, "vllm": "0.30.0"}
    elif note == "vLLM 심화 노트":                # 용량 보고서 형식
        data = {"조건": {"모델": M15, "GPU": "Tesla T4", "vllm": "0.30.0", "입력_토큰": 512, "출력_토큰": 128},
                "손잡이": {"max_num_seqs": "기본(256)", "gpu_memory_utilization": "기본(0.92)"},
                "세_숫자": {"계산한_동시수": 40, "엔진이_잡은_동시수": 36, "약속을_지킨_동시수": 16},
                "서비스_한계": 16, "선점": 0, "접두사_캐시_적중률_A": 0.62}
    elif note == "TensorRT-LLM 노트":             # CSV 형식. 행마다 조건이 있습니다
        with open(path, "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow(["run_id", "source", "engine", "engine_version", "model", "precision", "gpu", "prompt_sha",
                        "concurrency", "output_tokens", "ttft_p99_ms", "tpot_p99_ms", "output_tok_s", "req_s", "errors"])
            for engine, ver, sp in [("vllm", "0.30.0", 1.0), ("trtllm", "1.2.1", 1.15)]:
                for n in (1, 8):
                    w.writerow([f"{engine}-{n}", "대역", engine, ver, M15, "fp16", "NVIDIA L4", "3f9a1c2b", n, 128,
                                round(180 * n ** 0.5 / sp), round(28 * n ** 0.3 / sp), round(70 * n ** 0.8 * sp, 1),
                                round(0.5 * n ** 0.8 * sp, 2), 0])
        return path
    else:                                          # RAG 서빙 최적화 노트 : meta 와 runs 형식
        runs = {}
        for label, keep, mt, layout, total in [("기준선", 5, 300, "good-score", 6200), ("고른 설정", 3, 120, "good-id", 3100)]:
            rows = [{"qid": f"Q{i:02d}", "q": f"(대역 질문 {i:02d})", "ok": i <= (20 if label == "기준선" else 22), "hit": True}
                    for i in range(1, 26)]
            runs[label] = {"cfg": {"candidates": 20, "keep": keep, "max_tokens": mt, "layout": layout, "route": "big"},
                           "summary": {"전체": [total, round(total * 1.4, 2)], "정답": sum(r["ok"] for r in rows),
                                       "적중": 25, "n": 25}, "flips": [], "rows": rows}
        data = {"meta": {"gen": "fake", "embedding": "intfloat/multilingual-e5-small", "reranker": "BAAI/bge-reranker-v2-m3",
                         "big": "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4", "small": M15, "gpu": None, "questions": 25},
                "runs": runs}
    data["_standin"] = STANDIN_NOTE
    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=1), encoding="utf-8")
    return path


(WORK / "standin").mkdir(exist_ok=True)
for note, places in SOURCES.items():
    if find_file(places) is None:
        print(f"{note:<14} mock :", make_standin(note, WORK / "standin" / places[0]))
print("예제 준비 완료. 다음 파일 확인 셀과 Step 2로 진행합니다.")

양자화 노트         mock : /content/d66/sample/standin/d54_results.json
프루닝과 증류 노트     mock : /content/d66/sample/standin/d55_results.json
GPTQ·AWQ 노트    mock : /content/d66/sample/standin/d56_results.json
vLLM 기초 노트     mock : /content/d66/sample/standin/d57_results.json
vLLM 심화 노트     mock : /content/d66/sample/standin/d58_capacity.json
TensorRT-LLM 노트 mock : /content/d66/sample/standin/d59_bench_mine.csv
RAG 서빙 최적화 노트  mock : /content/d66/sample/standin/d60_results.json
예제 준비 완료. 다음 파일 확인 셀과 Step 2로 진행합니다.


In [5]:
M7 = "Qwen/Qwen2.5-7B-Instruct"        # vLLM 기초 노트의 transformers 기준선 모델
FILES = {}                             # 노트 이름 -> (경로, 모드)
for note, places in SOURCES.items():
    found, mock = find_file(places), WORK / "standin" / places[0]
    if found or mock.exists():
        FILES[note] = (found, "실측") if found else (mock, "대역")
        print(f"{note:<14} {FILES[note][1]:<4} {FILES[note][0]}")
    else:
        print(f"{note:<14} 없음 : 바로 앞의 모의 결과 생성 셀을 실행해주세요")
n_real = sum(kind == "실측" for _, kind in FILES.values())
print(f"\n실측 {n_real}개, mock {len(FILES) - n_real}개, 없음 {len(SOURCES) - len(FILES)}개")

# 이전 교안을 열지 않고 이 출력과 Step 2 설명으로 원본 구조를 확인합니다.
rag_path = FILES["RAG 서빙 최적화 노트"][0]
rag_example = json.loads(rag_path.read_text(encoding="utf-8"))
print("\n[RAG 원본의 meta]", rag_example.get("meta", {}))
for label, run in rag_example.get("runs", {}).items():
    print(label, "/ cfg:", run.get("cfg"), "/ summary:", run.get("summary"))
    print("문항별 결과 예:", run.get("rows", [])[:1])

양자화 노트         대역   /content/d66/sample/standin/d54_results.json
프루닝과 증류 노트     대역   /content/d66/sample/standin/d55_results.json
GPTQ·AWQ 노트    대역   /content/d66/sample/standin/d56_results.json
vLLM 기초 노트     대역   /content/d66/sample/standin/d57_results.json
vLLM 심화 노트     대역   /content/d66/sample/standin/d58_capacity.json
TensorRT-LLM 노트 대역   /content/d66/sample/standin/d59_bench_mine.csv
RAG 서빙 최적화 노트  대역   /content/d66/sample/standin/d60_results.json

실측 0개, mock 7개, 없음 0개

[RAG 원본의 meta] {'gen': 'fake', 'embedding': 'intfloat/multilingual-e5-small', 'reranker': 'BAAI/bge-reranker-v2-m3', 'big': 'Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4', 'small': 'Qwen/Qwen2.5-1.5B-Instruct', 'gpu': None, 'questions': 25}
기준선 / cfg: {'candidates': 20, 'keep': 5, 'max_tokens': 300, 'layout': 'good-score', 'route': 'big'} / summary: {'전체': [6200, 8680.0], '정답': 20, '적중': 25, 'n': 25}
문항별 결과 예: [{'qid': 'Q01', 'q': '(대역 질문 01)', 'ok': True, 'hit': True}]
고른 설정 / cfg: {'candidates': 20, 'keep': 3, 'max_toke

## Step 2. 결과 행의 형식 통일

오늘 만든 원본을 `params`(바꾼 설정), `metrics`(수치와 단위), `tags`(질문 세트·모델·GPU 등 조건), `file`(첨부할 원본)로 정리합니다. 원본에 없는 값은 `빠짐`으로 기록합니다. 현재 CPU 런타임의 정보를 과거 측정 환경에 채워 넣으면 안 됩니다.

RAG 예제의 `meta`는 환경 정보, `runs`는 기준선·후보별 결과입니다. `cfg.keep`는 제공할 문서 조각 수, `max_tokens`는 답 길이 상한입니다. `summary.정답`·`n`은 맞힌 수·전체 문항 수, `summary.전체`는 p50·p95 지연 <strong>밀리초(ms)</strong>입니다. `rows`에는 문항별 `qid`·질문 `q`·정답 여부 `ok`·검색 적중 여부 `hit`가 들어 있습니다. 읽기 함수는 지연을 <strong>초(s)</strong>로 환산합니다.

`rows_from_d60()`의 표시된 두 줄에서 원본의 환경을 `tags`로 옮깁니다. 코드와 바로 아래 풀이를 함께 제공하며, 이전 교안이나 측정 파일을 열지 않아도 원본 출력과 이 설명으로 확인할 수 있습니다. 예제 기준선의 p50은 6.2초, 고른 설정은 3.1초로 기록됩니다.

In [6]:
REQUIRED = ["gpu", "model", "qset", "versions"]  # 이론 2.2절의 실행 조건입니다. 확인 못 한 값은 빠짐으로 남깁니다.


def qset_digest(items):
    """질문 세트의 해시(질문 문장이 없으면 질문 번호로)"""
    items = sorted(set(items))
    return hashlib.sha256("\n".join(items).encode("utf-8")).hexdigest()[:12] if items else None


def count_flips(base, other):
    """원본이 맞히고 경량화한 모델이 틀린 질문(flip)과 그 반대의 수(양자화 노트의 용어)"""
    common = set(base) & set(other)
    return {"lost": sum(base[q] and not other[q] for q in common),
            "gained": sum(other[q] and not base[q] for q in common)} if common else None


def make_row(note, run_name, params, metrics, tags, file, source):
    tags = {k: v for k, v in tags.items() if v not in (None, "", {})}
    missing = [k for k in REQUIRED if k not in tags]
    tags.update({"note": note, "source": source, "file": Path(file).name, "missing": ",".join(missing) or "없음",
                 **{k: "빠짐" for k in missing}})
    metrics = {k: float(v) for k, v in metrics.items() if isinstance(v, (int, float)) and not isinstance(v, bool)}
    return {"note": note, "run_name": run_name, "params": {k: str(v) for k, v in params.items()},
            "metrics": metrics, "tags": {k: str(v) for k, v in tags.items()}, "file": str(file), "missing": missing}

In [ ]:
def versions_text(d):
    d = {k: v for k, v in (d or {}).items() if v}
    return ", ".join(f"{k} {v}" for k, v in sorted(d.items())) or None


def rows_from_measure(note, path, source):
    """양자화, 프루닝과 증류, GPTQ·AWQ 노트 : 행 이름 -> 하네스 결과, _conditions(없을 수도 있음)"""
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    cond = data.get("_conditions", {})
    lines = {k: v for k, v in data.items() if not k.startswith("_") and isinstance(v, dict) and "correct" in v}
    base = lines.get("16bit")
    rows = []
    for name, r in lines.items():
        metrics = {"correct": sum(r["correct"].values()), "questions": len(r["correct"]),
                   "tokens_per_s": r.get("tps"), "peak_gb": r.get("peak_gb"), "weights_gb": r.get("weights_gb")}
        flips = count_flips(base["correct"], r["correct"]) if base else None
        if flips:
            metrics.update({"flips_lost": flips["lost"], "flips_gained": flips["gained"]})
        tags = {"gpu": cond.get("gpu"), "model": r.get("model_id") or cond.get("model"),
                "qset": qset_digest(r["correct"]), "versions": versions_text(cond.get("versions"))}
        rows.append(make_row(note, f"{note} · {name}", {"precision": name}, metrics, tags, path, source))
    return rows


def rows_from_d57(note, path, source):
    """vLLM 기초 노트(RunPod) : 30문항 행, 팀원마다 기동한 서버("팀 설정 측정"), "용량" """
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    cond = data.get("_conditions", {})
    base = {"gpu": cond.get("gpu"), "versions": versions_text({"vllm": cond.get("vllm")}), "team": cond.get("team")}
    lines = {n: r for n, r in data.items() if not n.startswith("_") and isinstance(r, dict) and "correct" in r}
    lines.update({f"팀 {n}": r for n, r in data.get("팀 설정 측정", {}).items()})
    rows = []
    for name, r in lines.items():
        # transformers 기준선은 7B 입니다. 조건의 model 은 서빙한 32B 4비트입니다
        model = r.get("모델") or r.get("model") or (M7 if name.startswith("transformers") else cond.get("model"))
        c = r.get("correct") or {}
        metrics = {"correct": r.get("정답"), "questions": len(c) or None, "wall_s": r.get("경과 초"),
                   "tokens_per_s": r.get("초당 토큰") or r.get("혼자 초당 토큰"),
                   "slowest_s": r.get("가장 늦은 사람 초") or r.get("최대 지연 초"),
                   "kv_tokens": r.get("kv_tokens"), "max_concurrency": r.get("max_concurrency")}
        params = {k: r[k] for k in ("engine", "역할", "설정", "비율", "max_model_len") if k in r}
        tags = {**base, "model": model, "qset": qset_digest(c)}
        rows.append(make_row(note, f"{note} · {name}", params, metrics, tags, path, source))
    cap = data.get("용량")
    if cap:                                        # 부하 측정이라 질문 세트가 없습니다
        metrics = {"vllm_limit": cap.get("vLLM 응답 시간 기준"), "hf_estimate": cap.get("transformers 어림"),
                   "vram_limit": cap.get("VRAM 기준")}
        rows.append(make_row(note, f"{note} · 용량", {"slo_s": cap.get("SLO 초")}, metrics,
                             {**base, "model": cond.get("model"), "qset": None}, path, source))
    return rows


def rows_from_d58(note, path, source):
    rep = json.loads(Path(path).read_text(encoding="utf-8"))
    cond = rep.get("조건", {})
    metrics = {**rep.get("세_숫자", {}), "service_limit": rep.get("서비스_한계"), "preempt": rep.get("선점"),
               "prefix_hit_A": rep.get("접두사_캐시_적중률_A")}
    tags = {"gpu": cond.get("GPU"), "model": cond.get("모델"), "qset": None,       # 부하 테스트라 질문 세트가 없습니다
            "versions": versions_text({"vllm": cond.get("vllm")})}
    params = {**rep.get("손잡이", {}), "input_tokens": cond.get("입력_토큰"), "output_tokens": cond.get("출력_토큰")}
    return [make_row(note, f"{note} · 용량 보고서", params, metrics, tags, path, source)]


def rows_from_d59(note, path, source):
    rows = []
    with open(path, encoding="utf-8") as f:
        for r in csv.DictReader(f):
            num = {k: float(r[k]) for k in ["ttft_p99_ms", "tpot_p99_ms", "output_tok_s", "req_s", "errors"] if r.get(k)}
            tags = {"gpu": r.get("gpu"), "model": r.get("model"), "qset": r.get("prompt_sha"),
                    "versions": f"{r.get('engine')} {r.get('engine_version')}"}
            row_source = "대역" if (source == "대역" or r.get("source") == "대역") else "실측"
            params = {k: r.get(k) for k in ("engine", "precision", "concurrency", "output_tokens")}
            rows.append(make_row(note, f"{note} · {r.get('run_id')}", params, num, tags, path, row_source))
    return rows

In [8]:
# 실습 : d60_results.json 을 읽어 실행마다 make_row(...) 로 행 하나를 만드세요

def rows_from_d60(note, path, source):
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    meta = data.get("meta", {})
    rows = []
    for label, run in data.get("runs", {}).items():
        s = run.get("summary") or {}
        total = s.get("전체") or [None, None]
        # D60 원본의 밀리초를 초로 환산해 응답 지연을 같은 단위로 읽습니다.
        metrics = {"correct": s.get("정답"), "hit": s.get("적중"), "questions": s.get("n"),
                   "total_p50_s": total[0]/1000 if total[0] is not None else None,
                   "total_p95_s": total[1]/1000 if total[1] is not None else None,
                   "flips_lost": len(run["flips"]) if isinstance(run.get("flips"), list) else None}
        qset = qset_digest(f"{x.get('qid')}|{x.get('q')}" for x in run.get("rows", []))
        ### START CODE HERE ### (조건 필드 tags 만들기 2줄)
        # 1) GPU 는 meta 의 gpu, 모델은 meta 의 big, 질문 세트는 qset. 버전은 파일에 없으므로 None 으로 둡니다(make_row 가 "빠짐"으로 표시)
        tags = {"gpu": meta.get("gpu"), "model": meta.get("big"), "qset": qset, "versions": None,
                "gen": meta.get("gen"), "date": meta.get("date")}
        ### END CODE HERE ###
        row_source = "대역" if (source == "대역" or data.get("_standin") or meta.get("standin") or meta.get("gen") == "fake") else "실측"
        rows.append(make_row(note, f"{note} · {label}", run.get("cfg", {}), metrics, tags, path, row_source))
    return rows

<details class="lms-answer">
<summary>풀이 보기</summary>

<strong>빈칸 정답 코드 (`### START CODE HERE ###` 영역)</strong>

```python
tags = {"gpu": meta.get("gpu"), "model": meta.get("big"), "qset": qset, "versions": None,
        "gen": meta.get("gen"), "date": meta.get("date")}
```

버전 필드는 이 파일에 없어서 "빠짐"으로 남습니다.

</details>

In [9]:
READERS = {"양자화 노트": rows_from_measure, "프루닝과 증류 노트": rows_from_measure, "GPTQ·AWQ 노트": rows_from_measure,
           "vLLM 기초 노트": rows_from_d57, "vLLM 심화 노트": rows_from_d58, "TensorRT-LLM 노트": rows_from_d59,
           "RAG 서빙 최적화 노트": rows_from_d60}
ROWS = []
for note, (path, kind) in FILES.items():
    got = READERS[note](note, path, kind)
    if got is None:
        print(f"{note} : 읽는 함수가 비어 있어 건너뜁니다. 실습 셀을 채우세요")
    ROWS.extend(got or [])
for r in ROWS:
    print(f"{r['run_name'][:34]:<34} {r['tags']['source']:<4} {', '.join(r['missing']) or '없음'}")
print(f"\n[측정 1] 행 {len(ROWS)}개 중 조건이 빠진 행 {sum(bool(r['missing']) for r in ROWS)}개")
print("빠진 필드별 :", dict(Counter(k for r in ROWS for k in r["missing"])) or "없음")

# 점검표(Step 7)가 읽을 수 있게 모은 결과를 파일로 남깁니다
_ = (WORK / "inventory.json").write_text(json.dumps(
    {"files": {n: {"path": str(p), "source": k} for n, (p, k) in FILES.items()}}, ensure_ascii=False, indent=1),
    encoding="utf-8")

양자화 노트 · 16bit                     대역   gpu
양자화 노트 · int8                      대역   gpu
양자화 노트 · nf4                       대역   gpu
프루닝과 증류 노트 · 16bit                 대역   없음
프루닝과 증류 노트 · 비정형 50%               대역   없음
프루닝과 증류 노트 · 증류 0.5B               대역   없음
GPTQ·AWQ 노트 · 16bit                대역   gpu, versions
GPTQ·AWQ 노트 · gptq                 대역   gpu, versions
GPTQ·AWQ 노트 · awq                  대역   gpu, versions
GPTQ·AWQ 노트 · 32B awq              대역   gpu, versions
vLLM 기초 노트 · transformers 16비트 한 명 대역   없음
vLLM 기초 노트 · vLLM 32B 4비트 한 명씩     대역   없음
vLLM 기초 노트 · 팀 A 기준                대역   없음
vLLM 기초 노트 · 팀 D 7B 16비트           대역   없음
vLLM 기초 노트 · 용량                    대역   qset
vLLM 심화 노트 · 용량 보고서                대역   qset
TensorRT-LLM 노트 · vllm-1           대역   없음
TensorRT-LLM 노트 · vllm-8           대역   없음
TensorRT-LLM 노트 · trtllm-1         대역   없음
TensorRT-LLM 노트 · trtllm-8         대역   없음
RAG 서빙 최적화 노트 · 기준선                대역   gpu, versions
RAG 서빙 최적화 노트 · 고른 설정              

**출력에서 확인**　`[측정 1]` 표에서 `빠짐`이 있는 실행 하나를 골라 원본 파일의 해당 항목을 확인합니다. 그 값이 없으면 어떤 비교를 보류해야 할지 적어 둡니다. 예를 들어 GPU를 확인할 수 없다면 다른 실행과 장비가 같았다고 판단할 수 없습니다.

## Step 3. 트래킹 서버 기동

**트래킹 서버**는 파이썬 코드에서 보낸 기록을 받아 저장하고, 조회 요청에 결과를 돌려주는 프로그램입니다. 서버 주소와 연결할 실험 이름을 정한 뒤 출력의 `실험`과 `서버 버전`을 확인합니다. 이 단계에서 준비하는 것은 기록을 받을 곳입니다.

이론 노트 2장의 **MLflow에서 관리하는 것**에 나온 구성을 사용합니다. 이 런타임의 SQLite 파일에는 실행 조건과 지표가, 아티팩트 폴더에는 원본 결과 파일이 쌓입니다. 다음 Step에서 두 기록을 같은 실행에 연결합니다.

In [10]:
def pick_port(base, tries=10, taken=()):
    for port in range(base, base + tries):
        with socket.socket() as probe:
            if port not in taken and probe.connect_ex(("127.0.0.1", port)) != 0:
                return port
    return base


def wait_health(url, limit_s=60):
    """/health 가 200 을 반환하기까지 걸린 초. 기동되지 않으면 None"""
    t0 = time.perf_counter()
    while time.perf_counter() - t0 < limit_s:
        try:
            with urllib.request.urlopen(url + "/health", timeout=2) as r:
                if r.status == 200:
                    return round(time.perf_counter() - t0, 1)
        except OSError:
            time.sleep(0.5)
    return None


def start_server(db, artifacts, port, extra_env=None, extra_args=(), cwd=None):
    """트래킹 서버를 기동합니다(사용 통계는 끔). 로그는 db 이름의 .log"""
    cmd = [sys.executable, "-m", "mlflow", "server", "--backend-store-uri", f"sqlite:///{db}",
           "--artifacts-destination", str(artifacts), "--host", "127.0.0.1", "--port", str(port), "--workers", "1",
           *extra_args]
    env = {**os.environ, "MLFLOW_DISABLE_TELEMETRY": "true", **(extra_env or {})}
    return subprocess.Popen(cmd, stdout=open(Path(db).with_suffix(".log"), "w"), stderr=subprocess.STDOUT,
                            env=env, cwd=cwd, start_new_session=(os.name != "nt"))


def stop_server(proc):
    """이 노트가 기동한 서버와 자식 프로세스를 함께 종료합니다."""
    if proc is None or proc.poll() is not None:
        return
    if os.name == "nt":
        subprocess.run(["taskkill", "/PID", str(proc.pid), "/T", "/F"],
                       capture_output=True, timeout=15)
    else:
        import signal
        os.killpg(proc.pid, signal.SIGTERM)
    try:
        proc.wait(timeout=15)
    except subprocess.TimeoutExpired:
        if os.name == "nt":
            proc.kill()
        else:
            os.killpg(proc.pid, signal.SIGKILL)
        proc.wait(timeout=5)


SERVERS = {}                                     # 이름 -> 프로세스. Step 9에서 종료합니다
TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI")
if TRACKING_URI:
    print("사내 트래킹 서버를 씁니다 :", TRACKING_URI)
else:
    PORT = pick_port(int(os.environ.get("D66_PORT", "8660")))
    SERVERS["main"] = start_server(WORK / "mlflow.db", WORK / "artifacts", PORT)
    TRACKING_URI = f"http://127.0.0.1:{PORT}"
    took = wait_health(TRACKING_URI)
    if took is None:
        raise RuntimeError(f"트래킹 서버 기동 실패. {WORK / 'mlflow.log'}를 확인하고 이 셀부터 다시 실행하세요")
    print(f"트래킹 서버 : {TRACKING_URI} (기동까지 {took}초)")

트래킹 서버 : http://127.0.0.1:8662 (기동까지 6.1초)


In [11]:
import mlflow
from mlflow import MlflowClient

mlflow.set_tracking_uri(TRACKING_URI)
client = MlflowClient(TRACKING_URI)
EXP = os.environ.get("D66_EXPERIMENT", f"D66-기록연습-{INPUT_MODE}")
exp = mlflow.set_experiment(EXP)
print("실험 :", exp.name, "/ 번호", exp.experiment_id)
print("서버 버전 :", urllib.request.urlopen(TRACKING_URI + "/version", timeout=5).read().decode())

실험 : D66-기록연습-sample / 번호 1
서버 버전 : 3.16.1


In [12]:
# 서버 확인 도우미 : 트래킹 서버가 종료되어 있으면 다시 기동합니다
def ensure_mlflow():
    """트랙과 부록 셀이 처음에 부릅니다"""
    if SERVERS.get("main") is None or wait_health(TRACKING_URI, limit_s=2) is not None:
        return                                   # 사내 트래킹 서버를 쓰거나 서버가 살아 있습니다
    SERVERS["main"] = start_server(WORK / "mlflow.db", WORK / "artifacts", PORT)
    took = wait_health(TRACKING_URI)
    if took is None:
        raise RuntimeError(f"서버 재기동 실패. {WORK / 'mlflow.log'}를 확인하세요")
    print("트래킹 서버를 다시 기동했습니다 :", TRACKING_URI, "/ 기동까지", took, "초")

**로컬 작업** : 사내에서 팀이 함께 쓰는 트래킹 서버는 SQLite 대신 PostgreSQL 같은 데이터베이스에 기록을 두고 산출물은 사내 파일 저장소에 둡니다. 사내망에 열 때는 `--host`와 `--allowed-hosts`를 함께 적고 앞에 게이트웨이를 둡니다.

## Step 4. 실험 기록

앞 Step에서 만든 공통 형식의 결과 행 하나를 <strong>실행(run)</strong> 하나로 기록합니다. `start_run()`은 새 실행을 열고, 뒤의 `log_params()`·`log_metrics()`·`log_artifact()`는 설정·수치·원본 파일을 그 실행에 붙입니다.

이론 노트 **2.2절**의 실행 추적을 MLflow에 구현합니다. 지표와 원본 파일을 같은 실행에 연결해야 화면의 정답 수와 원본 JSON을 대조할 수 있습니다. `row_key`는 같은 결과 행을 재실행으로 여러 번 등록하는 일을 막습니다. 중복 행이 쌓이면 하나의 결과를 여러 차례 검증한 것처럼 읽을 수 있습니다.

두 번 연속 업로드한 뒤 두 번째 출력의 `기록`이 0이고 같은 행이 모두 `건너뜀`으로 집계됐는지 확인합니다. 이미 기록이 있는 실험에서는 첫 번째 업로드에서도 건너뛸 수 있습니다.

In [ ]:
def row_key(r):
    """행의 내용 해시. 실행 이름, 파라미터, 지표, 조건 태그, 원본 파일 이름이 같으면 같은 해시입니다"""
    body = json.dumps([r["run_name"], r["params"], r["metrics"], r["tags"], Path(r["file"]).name], ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(body.encode("utf-8")).hexdigest()[:16]


# 실습 : 같은 row_key 를 가진 실행이 이 실험에 이미 있으면 True, 없으면 False 를 반환하세요

def already_logged(key):
    ### START CODE HERE ### (row_key 가 같은 실행을 찾아 있는지 판정하기 2줄)
    # 1) search_runs 의 filter_string 에 tags.row_key 조건을 겁니다. 맞는 실행이 표(DataFrame)로 옵니다
    found = mlflow.search_runs(experiment_names=[EXP], filter_string=f"tags.row_key = '{key}'")
    # 2) 표의 행 수가 0 보다 크면 이미 기록한 행입니다
    found = len(found) > 0
    ### END CODE HERE ###
    return found

<details class="lms-answer">
<summary>풀이 보기</summary>

<strong>빈칸 정답 코드 (`### START CODE HERE ###` 영역)</strong>

```python
found = mlflow.search_runs(experiment_names=[EXP], filter_string=f"tags.row_key = '{key}'")
found = len(found) > 0
```

`search_runs`는 조건에 맞는 실행을 표로 반환합니다. 행 수가 0이면 처음 보는 행입니다.

</details>

In [14]:
def upload(rows):
    """행마다 실행 하나. 이미 있는 행은 건너뜁니다"""
    if already_logged("확인용") is None:
        print("already_logged 가 비어 있습니다. 실습 셀을 채우고 이 셀을 다시 실행하세요")
        return None
    t0 = time.perf_counter()
    added = skipped = 0
    for r in rows:
        key = row_key(r)
        # 다시 실행해도 같은 결과가 별개의 평가 기록으로 쌓이지 않게 합니다.
        if already_logged(key):
            skipped += 1
            continue
        with mlflow.start_run(run_name=r["run_name"]):
            mlflow.log_params(r["params"])
            mlflow.log_metrics(r["metrics"])
            mlflow.set_tags({**r["tags"], "row_key": key})
            # 실행 화면의 요약 지표를 원본과 대조할 수 있게 같은 실행에 파일을 붙입니다.
            mlflow.log_artifact(r["file"], artifact_path="raw")
        added += 1
    return added, skipped, round(time.perf_counter() - t0, 1)


first = upload(ROWS)
if first:
    print(f"첫 번째 : 기록 {first[0]}, 건너뜀 {first[1]}, {first[2]}초")
    second = upload(ROWS)
    print(f"두 번째 : 기록 {second[0]}, 건너뜀 {second[1]}, {second[2]}초")

첫 번째 : 기록 0, 건너뜀 22, 0.4초
두 번째 : 기록 0, 건너뜀 22, 0.4초


## Step 5. 기록 질의

점수를 읽기 전에 비교 조건을 확인합니다. `flips_lost`는 기준선이 맞힌 문항을 후보가 틀린 수입니다. 첫 조회는 `gpu` 태그가 T4이고 이 수가 0인 행을 처리량순으로 찾습니다. 예제에 적힌 T4는 모의 입력의 조건이며 현재 GPU를 기동한다는 뜻이 아닙니다.

출력의 `실측`·`대역`을 먼저 확인합니다. 기본 경로에서는 모두 `대역`이며, 검색 필터와 원본 확인 절차를 연습합니다.

In [15]:
runs = mlflow.search_runs(experiment_names=[EXP])
KIND = "실측" if len(runs) and (runs["tags.source"] == "실측").any() else "대역"
t0 = time.perf_counter()
best = mlflow.search_runs(
    experiment_names=[EXP],
    filter_string=f"tags.source = '{KIND}' and tags.gpu LIKE '%T4%' and metrics.flips_lost = 0",
    order_by=["metrics.tokens_per_s DESC"], max_results=3)
print(f"실행 {len(runs)}개 / [측정 2] 질문 하나에 {(time.perf_counter() - t0) * 1000:.0f}ms ({KIND} 행에서 찾음)")
if len(runs) == 0:
    print("트래킹 서버에 실행이 없습니다. Step 4의 실습 셀을 채우고 기록하는 셀부터 다시 실행하세요")
else:
    print(best[["tags.mlflow.runName", "metrics.tokens_per_s", "metrics.correct", "tags.gpu"]].to_string(index=False)
          if len(best) else "조건에 맞는 실행이 없습니다")

실행 22개 / [측정 2] 질문 하나에 28ms (대역 행에서 찾음)
tags.mlflow.runName  metrics.tokens_per_s  metrics.correct tags.gpu
 프루닝과 증류 노트 · 16bit                  21.0             27.0 Tesla T4


**출력에서 확인**　T4 조건에 맞는 실행이 없다면 GPU 태그와 출처를 확인합니다. GPU가 `빠짐`인 행은 기록에 남고, 이 조회의 후보에서는 빠집니다.

이어서 두 쌍을 비교합니다. `compare()`는 질문 세트 해시·모델·GPU가 각각 일치하고 세 조건이 모두 기록된 실행을 통과시킵니다. 질문이 다르면 정답 수가 달라진 이유를 구분하기 어렵고, GPU가 다르면 속도 차이를 설정의 효과로 해석하기 어렵습니다. 이 검사는 비교의 출발점입니다. 실제 성능을 판단할 때는 원본의 실행 모드·채점 기준·동시 요청 조건도 확인합니다.

일부 예제의 `qset`은 질문 번호 목록만 해시합니다. 번호가 같아도 본문이나 채점 기준이 다를 수 있으므로, 실제 결과를 비교할 때는 질문 원본과 판정 기준까지 대조해야 합니다. 여기서는 예제 안에서 같은 문항 집합으로 만든 두 행과 조건이 빠진 행을 구분합니다.

In [16]:
GROUP = ["tags.qset", "tags.model", "tags.gpu"]
if len(runs) == 0:
    print("실행이 없어 묶을 것이 없습니다")
else:
    for key, g in runs.groupby(GROUP):
        flag = "  (빠진 조건이 있어 비교하지 않음)" if "빠짐" in key else ""
        print(f"{len(g):>2}개 | 질문 세트 {key[0][:12]} | {key[1].split('/')[-1][:24]} | {key[2]}{flag}")

 1개 | 질문 세트 3aec178a3dbd | 7b-awq-ko | 빠짐  (빠진 조건이 있어 비교하지 않음)
 1개 | 질문 세트 3aec178a3dbd | 7b-gptq-ko | 빠짐  (빠진 조건이 있어 비교하지 않음)
 1개 | 질문 세트 3aec178a3dbd | Qwen2.5-0.5B-Instruct | Tesla T4
 2개 | 질문 세트 3aec178a3dbd | Qwen2.5-1.5B-Instruct | Tesla T4
 3개 | 질문 세트 3aec178a3dbd | Qwen2.5-1.5B-Instruct | 빠짐  (빠진 조건이 있어 비교하지 않음)
 2개 | 질문 세트 3aec178a3dbd | Qwen2.5-32B-Instruct-AWQ | NVIDIA L40S
 1개 | 질문 세트 3aec178a3dbd | Qwen2.5-32B-Instruct-AWQ | 빠짐  (빠진 조건이 있어 비교하지 않음)
 2개 | 질문 세트 3aec178a3dbd | Qwen2.5-7B-Instruct | NVIDIA L40S
 1개 | 질문 세트 3aec178a3dbd | Qwen2.5-7B-Instruct | 빠짐  (빠진 조건이 있어 비교하지 않음)
 4개 | 질문 세트 3f9a1c2b | Qwen2.5-1.5B-Instruct | NVIDIA L4
 2개 | 질문 세트 be85881f3606 | Qwen2.5-7B-Instruct-GPTQ | 빠짐  (빠진 조건이 있어 비교하지 않음)
 1개 | 질문 세트 빠짐 | Qwen2.5-1.5B-Instruct | Tesla T4  (빠진 조건이 있어 비교하지 않음)
 1개 | 질문 세트 빠짐 | Qwen2.5-32B-Instruct-AWQ | NVIDIA L40S  (빠진 조건이 있어 비교하지 않음)


In [17]:
# 실습 : 고른 실행들이 같은 비교 그룹(질문 세트 해시, 모델, GPU가 모두 같고 "빠짐"이 없음)에 있는지 판정하세요

def compare(names, metric="metrics.correct"):
    sub = runs[runs["tags.mlflow.runName"].isin(names)]
    if len(sub) != len(names):
        return f"찾지 못한 실행이 있습니다 : {sorted(set(names) - set(sub['tags.mlflow.runName']))}"
    ### START CODE HERE ### (GROUP 의 필드마다 값이 하나뿐이고 "빠짐"이 없는지 판정하기 1줄)
    # 같은 조건끼리 비교합니다. 두 실행에 모두 GPU가 빠져 있어도 같은 장비에서 쟀다고 볼 수 없습니다.
    same = all(sub[c].nunique() == 1 for c in GROUP) and not (sub[GROUP] == "빠짐").any().any()
    ### END CODE HERE ###
    if same is None:
        return "same 이 비어 있습니다. 실습 셀을 채우세요"
    if not same:
        return "비교 그룹이 다릅니다. 이 실행들은 비교하지 않습니다 : " + "; ".join(
            f"{r['tags.mlflow.runName']} ({r['tags.qset'][:8]}, {r['tags.gpu']})" for _, r in sub.iterrows())
    return sub[["tags.mlflow.runName", metric]].to_string(index=False)

<details class="lms-answer">
<summary>풀이 보기</summary>

<strong>빈칸 정답 코드 (`### START CODE HERE ###` 영역)</strong>

```python
same = all(sub[c].nunique() == 1 for c in GROUP) and not (sub[GROUP] == "빠짐").any().any()
```

필드마다 값이 하나뿐이고 "빠짐"이 없어야 같은 그룹입니다.

</details>

In [18]:
CASES = {
    "같은 하네스, 같은 7B, 같은 Pod": ["vLLM 기초 노트 · transformers 16비트 한 명씩", "vLLM 기초 노트 · 팀 D 7B 16비트"],
    "GPU가 빠진 행과 비교": ["GPTQ·AWQ 노트 · 16bit", "vLLM 기초 노트 · transformers 16비트 한 명씩"],
}
for title, names in CASES.items():
    print(f"[{title}]")
    print(compare(names) if len(runs) else "실행이 없어 비교할 것이 없습니다", "\n")

[같은 하네스, 같은 7B, 같은 Pod]
                tags.mlflow.runName  metrics.correct
           vLLM 기초 노트 · 팀 D 7B 16비트             29.0
vLLM 기초 노트 · transformers 16비트 한 명씩             29.0 

[GPU가 빠진 행과 비교]
비교 그룹이 다릅니다. 이 실행들은 비교하지 않습니다 : vLLM 기초 노트 · transformers 16비트 한 명씩 (3aec178a, NVIDIA L40S); GPTQ·AWQ 노트 · 16bit (3aec178a, 빠짐) 



**출력에서 확인**　예제에서 `같은 하네스, 같은 7B, 같은 Pod` 쌍은 공통 질문 번호·모델·GPU 태그를 갖고, `GPU가 빠진 행과 비교` 쌍은 비교가 보류됩니다. 이름에 있는 하네스는 같은 질문을 실행·채점하는 프로그램을 뜻하며 이 실습에서 가져오거나 실행하지 않습니다. 보류한 쌍에서 무엇을 더 확인해야 할지 원본 필드로 설명해주세요. 이 출력은 모의 데이터의 기록 절차이며 실제 엔진의 우열을 검증한 결과는 아닙니다.

## Step 6. 서비스 버전 등록과 별칭

**모델 레지스트리**는 이름별 버전과 출처를 보관하는 목록입니다. Step 4의 실행은 평가 기록이고, 여기서는 그 실행을 특정 서비스 구성의 버전에 연결합니다. `운영`은 현재 선택한 버전, `직전`은 되돌릴 버전을 가리키는 별칭입니다.

오늘 생성한 RAG 예제의 `기준선`과 `고른 설정`을 각각 등록하고 `source_run`으로 평가 실행을 연결합니다. 모델 카드는 설정·조건·결과·출처를 적은 설명문이며 아래 `card_for()`가 만듭니다. 예전 모델 카드나 실제 챗봇 서버는 필요하지 않습니다.

두 버전 번호와 별칭을 확인한 뒤 `rollback()`으로 선택을 되돌립니다. 이는 모의 결과로 버전·출처를 관리하는 연습입니다. 실제 서비스의 모델 로딩·정상 응답 복구는 별도 검증이 필요합니다.

In [19]:
from mlflow.exceptions import MlflowException

NAME = os.environ.get("D66_MODEL_NAME", "사내규정-RAG")
PROMPT_VERSION = os.environ.get("D66_PROMPT_VERSION", "D66-예제-v1")    # 별도 파일 없이 쓰는 예제 프롬프트 식별자
d60_file = FILES.get("RAG 서빙 최적화 노트", (None,))[0]
d60_meta = json.loads(Path(d60_file).read_text(encoding="utf-8")).get("meta", {}) if d60_file else {}


def find_run(run_name):
    df = mlflow.search_runs(experiment_names=[EXP], filter_string=f"tags.mlflow.runName = '{run_name}'")
    return None if len(df) == 0 else df.iloc[0]


def aliases_of(name=NAME):
    """별칭 -> 버전 번호. 등록된 모델이 아직 없으면 빈 dict"""
    try:
        return client.get_registered_model(name).aliases
    except MlflowException:
        return {}


def card_for(label, p, run):
    """이 셀에서 서비스 설정·평가 조건·출처를 담은 카드를 만듭니다"""
    m = {k: run.get(f"metrics.{k}") for k in ["correct", "questions", "hit", "total_p50_s", "total_p95_s"]}
    m = {k: "없음" if v is None or v != v else f"{v:g}" for k, v in m.items()}
    return "\n".join([
        f"# {NAME} : {label}",
        f"- 원본 : 큰 모델 {p.get('llm')} / 작은 모델 {p.get('small_llm')} / 임베딩 {p.get('embedding')}",
        f"- 서비스 설정 : 후보 {p.get('candidates')}, 넣는 청크 {p.get('keep')}, 답 길이 상한 {p.get('max_tokens')}, "
        f"순서 {p.get('layout')}, 프롬프트 버전 {p.get('prompt_version')}",
        f"- 잰 조건 : GPU {run['tags.gpu']} / 질문 세트 {run['tags.qset']} / 모드 {run['tags.source']} / 버전 {run['tags.versions']}",
        f"- 결과 : 정답 {m['correct']}/{m['questions']}, 적중 {m['hit']}, 전체 p50 {m['total_p50_s']}초, p95 {m['total_p95_s']}초",
        f"- 잰 실행 : {run['run_id']}",
    ])


existing = {mv.tags.get("source_run"): mv.version for mv in client.search_model_versions(f"name='{NAME}'")}
VERSIONS = {}
for label in ["기준선", "고른 설정"]:
    run = find_run(f"RAG 서빙 최적화 노트 · {label}")
    if run is None:
        print(f"{label} 실행이 없습니다. Step 2의 실습을 채우고 Step 4까지 다시 실행하세요")
    elif run["run_id"] in existing:
        VERSIONS[label] = existing[run["run_id"]]
    else:
        params = {k[7:]: v for k, v in run.items() if k.startswith("params.") and v is not None}
        params.update({"llm": d60_meta.get("big"), "small_llm": d60_meta.get("small"), "embedding": d60_meta.get("embedding"),
                       "reranker": d60_meta.get("reranker"), "prompt_version": PROMPT_VERSION})
        # 장애 조사 때 서비스 버전에서 그 버전을 평가한 실행을 찾을 수 있게 연결합니다.
        lm = mlflow.create_external_model(name="rag-service", params={k: str(v) for k, v in params.items()},
                                          model_type="rag-service", source_run_id=run["run_id"])
        mv = mlflow.register_model(f"models:/{lm.model_id}", NAME, tags={"source_run": run["run_id"], "label": label})
        client.update_model_version(NAME, mv.version, description=card_for(label, params, run))
        VERSIONS[label] = mv.version

print("등록된 버전 :", VERSIONS)
if {"기준선", "고른 설정"} <= set(VERSIONS):
    client.set_registered_model_alias(NAME, "운영", VERSIONS["고른 설정"])
    client.set_registered_model_alias(NAME, "직전", VERSIONS["기준선"])
print("별칭 :", aliases_of())

등록된 버전 : {'기준선': '1', '고른 설정': '2'}
별칭 : {'운영': '2', '직전': '1'}


![운영 별칭의 변경과 서버의 설정 반영은 별도 단계입니다. 별칭을 다시 읽은 뒤의 응답 버전을 확인합니다.](lms-img:0)

*롤백 연습에서는 `운영` 별칭을 이전 버전으로 옮깁니다. 실제 서비스에서는 서버가 해당 설정을 읽은 뒤 정상 요청에 응답하는지 확인해야 합니다.*

In [20]:
# 실습 : "운영"을 "직전"이 가리키던 버전으로, "직전"을 원래 "운영"이 가리키던 버전으로 맞바꾸세요

def rollback(name=NAME):
    aliases = aliases_of(name)                                   # 별칭 -> 버전 번호 (Step 6의 함수)
    if "운영" not in aliases or "직전" not in aliases:
        print("운영과 직전 별칭이 모두 있어야 롤백할 수 있습니다 :", aliases)
        return None
    t0 = time.perf_counter()
    cur, prev = aliases["운영"], aliases["직전"]
    ### START CODE HERE ### (운영과 직전 별칭을 맞바꾸기 2줄)
    # 새 설정에 문제가 생기면 운영 별칭을 이전에 쓰던 버전으로 돌립니다.
    client.set_registered_model_alias(name, "운영", prev)
    # 다시 앞으로 옮기는 연습에 쓸 수 있도록 방금 내린 버전도 직전 별칭에 남깁니다.
    client.set_registered_model_alias(name, "직전", cur)
    ### END CODE HERE ###
    swapped = {"운영": prev, "직전": cur, "ms": round((time.perf_counter() - t0) * 1000, 1)}
    return swapped

<details class="lms-answer">
<summary>풀이 보기</summary>

<strong>빈칸 정답 코드 (`### START CODE HERE ###` 영역)</strong>

```python
client.set_registered_model_alias(name, "운영", prev)
client.set_registered_model_alias(name, "직전", cur)
```

버전을 새로 만들거나 지우지 않고 별칭 두 개만 옮깁니다.

</details>

In [21]:
# 롤백을 연습하고 다시 앞으로 옮깁니다. 남긴 기록은 점검표(Step 7) 9번의 증거입니다
EVID = WORK / "evidence"
EVID.mkdir(exist_ok=True)
moved = rollback()
if moved is None:
    print("rollback 이 비어 있거나 별칭이 없습니다. 실습 셀을 채우고 이 셀을 다시 실행하세요")
else:
    # 함수가 의도한 값과 레지스트리에서 다시 읽은 실제 별칭을 함께 봅니다.
    print(f"[측정 3] 별칭을 옮기는 데 {moved['ms']}ms : {moved} / 지금 별칭 {aliases_of()}")
    back = rollback()
    print("다시 앞으로 :", back)
    _ = (EVID / "rollback.json").write_text(json.dumps(
        {"date": time.strftime("%Y-%m-%d %H:%M"), "moved": moved, "back": back}, ensure_ascii=False, indent=1),
        encoding="utf-8")

[측정 3] 별칭을 옮기는 데 52.4ms : {'운영': '1', '직전': '2', 'ms': 52.4} / 지금 별칭 {'운영': '1', '직전': '2'}
다시 앞으로 : {'운영': '2', '직전': '1', 'ms': 51.6}


**출력에서 확인**　처음의 `별칭`과 `[측정 3]`의 `지금 별칭`을 대조해 `운영`이 처음의 `직전` 버전을 가리키는지 봅니다. 밀리초로 출력된 값은 레지스트리에서 별칭을 바꾸는 데 걸린 시간입니다. 실제 서비스 복구를 확인하려면 정상 요청을 보내 기대한 버전과 응답을 확인합니다. 부록 실습 A에서 별칭을 다시 읽는 모의 서버로 이 차이를 살펴봅니다.

## Step 7. 수준 점검표 채점기 (선택)

당일 본 실습은 Step 6 뒤 Step 9로 이어집니다. 이 채점기는 성숙도 판정을 더 연습할 때 씁니다.
관련 설명은 이론 노트 4.1절에 포함되어 있습니다.

이론 노트 4.1절의 13항목을 그대로 채점기로 만듭니다. 오늘 실습 환경에는 다른 사람의 재실행 기록과 파이프라인·배포·모니터링의 근거 파일이 아직 없습니다. 따라서 채점기는 7번, 11번, 12번, 13번을 `불합격`으로 출력할 수 있습니다. 이때는 실제 기능이 실패한 것인지, 아직 실행하지 않아 근거가 없는 것인지 구분해 기록합니다. 사람이 남기는 증거(다른 사람이 처음부터 다시 만든 기록, 파이프라인, 배포, 경보)는 `evidence` 폴더의 파일로 받습니다. 10번(답마다 버전)은 부록 실습 A의 요청 기록이 있어야 통과합니다.

In [22]:
%%writefile level_check.py
"""수준 점검표 채점기 (MLOps 소개 이론 노트 4.1절의 13항목). 증거를 확인하지 못하면 불합격입니다.
환경 변수 : MLFLOW_TRACKING_URI, D66_EXPERIMENT, D66_MODEL_NAME, D66_WORK
"""
import json
import os
from pathlib import Path

os.environ.setdefault("MLFLOW_DISABLE_TELEMETRY", "true")
import mlflow
from mlflow import MlflowClient

URI = os.environ["MLFLOW_TRACKING_URI"]
EXP = os.environ.get("D66_EXPERIMENT", "서빙 심화 주간")
NAME = os.environ.get("D66_MODEL_NAME", "사내규정-RAG")
WORK = Path(os.environ.get("D66_WORK", "/content/d66"))
mlflow.set_tracking_uri(URI)
client = MlflowClient(URI)
runs = lambda: mlflow.search_runs(experiment_names=[EXP])


def filled(*cols):
    """cols 필드가 모두 있고 "빠짐"이 아닌 실행만 통과입니다"""
    df = runs()
    bad = [i for i, r in df.iterrows() if any(c not in df or r[c] != r[c] or r[c] == "빠짐" for c in cols)]
    return len(df) > 0 and not bad, f"{'·'.join(c[5:] for c in cols)} 필드가 빠진 실행 {len(bad)}/{len(df)}"


def evidence(fname, where):
    ok = (WORK / "evidence" / fname).exists()
    return ok, f"{fname} 있음" if ok else f"증거 파일 {fname} 없음 ({where})"


def c_one_place():
    inv = json.loads((WORK / "inventory.json").read_text(encoding="utf-8"))
    lacking = {Path(v["path"]).name for v in inv["files"].values()} - set(runs()["tags.file"].dropna())
    return not lacking, f"실행이 없는 파일 {sorted(lacking)}" if lacking else "결과 파일마다 실행이 있음"


def c_alias():
    aliases = client.get_registered_model(NAME).aliases
    return "운영" in aliases, f"별칭 {aliases}"


def c_cards():
    mvs = client.search_model_versions(f"name='{NAME}'")
    bad = [mv.version for mv in mvs if not all(k in (mv.description or "") for k in ["원본", "잰 조건", "결과"])]
    return bool(mvs) and not bad, f"버전 {len(mvs)}개, 카드가 빈 버전 {bad}"


def c_trace():
    mv = client.get_model_version_by_alias(NAME, "운영")
    run_id = mlflow.get_logged_model(mv.source.split("/")[-1]).source_run_id
    if not run_id:
        return False, "버전에 측정한 실행이 이어져 있지 않음"
    card = client.get_run(run_id).data.tags.get("dataset_card")
    return bool(card), f"버전 {mv.version} → 실행 → 데이터셋 카드 {card or '태그(dataset_card)가 없음'}"


def c_ledger():
    p = WORK / "ledger.jsonl"
    lines = [json.loads(x) for x in p.read_text(encoding="utf-8").splitlines() if x.strip()] if p.exists() else []
    stamped = sum(bool(x.get("version")) for x in lines)
    return bool(lines) and stamped == len(lines), f"요청 {len(lines)}건 중 버전이 적힌 것 {stamped}건 (부록 실습 A)"


ITEMS = [
    (1, "버전을 고정해 설치한다", 1, lambda: filled("tags.versions")),
    (2, "결과마다 조건을 적는다", 2, lambda: filled("tags.gpu", "tags.model")),
    (3, "질문 세트를 고정한다", 2, lambda: filled("tags.qset")),
    (4, "실험 기록이 한곳에 있다", 2, c_one_place),
    (5, "운영 버전에 별칭이 있다", 2, c_alias),
    (6, "버전마다 카드가 있다", 2, c_cards),
    (7, "다른 사람이 처음부터 다시 만든다", 2, lambda: evidence("repro.json", "다른 사람의 계정에서 다시 만든 기록")),
    (8, "버전에서 측정과 데이터까지 추적한다", 3, c_trace),
    (9, "옛 버전으로 롤백한다", 3, lambda: evidence("rollback.json", "Step 6의 롤백 기록")),
    (10, "답마다 버전이 남는다", 3, c_ledger),
    (11, "다시 만드는 일이 파이프라인으로 실행된다", 2, lambda: evidence("pipeline.json", "파이프라인 정의와 실행 기록")),
    (12, "테스트를 거친 배포가 자동이다", 3, lambda: evidence("deploy.json", "테스트를 거친 자동 배포의 기록")),
    (13, "운영 지표가 다시 만드는 일을 트리거한다", 4, lambda: evidence("alert_to_run.json", "경보가 트리거한 실행의 기록")),
]


if __name__ == "__main__":
    res = []
    for no, title, level, check in ITEMS:
        try:
            ok, why = check()
            state = "통과" if ok else "불합격"
        except Exception as e:                      # 확인하지 못하면 불합격으로 셉니다
            state, why = "확인 못 함", f"{type(e).__name__}: {str(e)[:80]}"
        res.append({"no": no, "item": title, "level": level, "state": state, "evidence": why})
        print(f"{no:>2} [{state}] 수준 {level} {title} : {why}")
    (WORK / "level_check.json").write_text(json.dumps(res, ensure_ascii=False, indent=1), encoding="utf-8")
    print("저장 :", WORK / "level_check.json")

Overwriting level_check.py


In [23]:
env = {**os.environ, "MLFLOW_TRACKING_URI": TRACKING_URI, "D66_EXPERIMENT": EXP, "D66_MODEL_NAME": NAME,
       "D66_WORK": str(WORK)}
t0 = time.perf_counter()
out = subprocess.run([sys.executable, "level_check.py"], env=env, capture_output=True, text=True, timeout=300)
print(out.stdout or out.stderr[-2000:])
print(f"채점에 {time.perf_counter() - t0:.1f}초")
CHECK = json.loads((WORK / "level_check.json").read_text(encoding="utf-8")) if (WORK / "level_check.json").exists() else []

 1 [불합격] 수준 1 버전을 고정해 설치한다 : versions 필드가 빠진 실행 6/22
 2 [불합격] 수준 2 결과마다 조건을 적는다 : gpu·model 필드가 빠진 실행 9/22
 3 [불합격] 수준 2 질문 세트를 고정한다 : qset 필드가 빠진 실행 2/22
 4 [통과] 수준 2 실험 기록이 한곳에 있다 : 결과 파일마다 실행이 있음
 5 [통과] 수준 2 운영 버전에 별칭이 있다 : 별칭 {'운영': '2', '직전': '1'}
 6 [불합격] 수준 2 버전마다 카드가 있다 : 버전 3개, 카드가 빈 버전 ['3']
 7 [불합격] 수준 2 다른 사람이 처음부터 다시 만든다 : 증거 파일 repro.json 없음 (다른 사람의 계정에서 다시 만든 기록)
 8 [불합격] 수준 3 버전에서 측정과 데이터까지 추적한다 : 버전 2 → 실행 → 데이터셋 카드 태그(dataset_card)가 없음
 9 [통과] 수준 3 옛 버전으로 롤백한다 : rollback.json 있음
10 [통과] 수준 3 답마다 버전이 남는다 : 요청 12건 중 버전이 적힌 것 12건 (부록 실습 A)
11 [불합격] 수준 2 다시 만드는 일이 파이프라인으로 실행된다 : 증거 파일 pipeline.json 없음 (파이프라인 정의와 실행 기록)
12 [불합격] 수준 3 테스트를 거친 배포가 자동이다 : 증거 파일 deploy.json 없음 (테스트를 거친 자동 배포의 기록)
13 [불합격] 수준 4 운영 지표가 다시 만드는 일을 트리거한다 : 증거 파일 alert_to_run.json 없음 (경보가 트리거한 실행의 기록)
저장 : /content/d66/sample/level_check.json

채점에 2.3초


채점 결과를 바탕으로 연습용 수준을 계산합니다. 이론 노트 3.6절에서 설명했듯, 실제 팀에는 여러 수준의 특징이 함께 나타날 수 있습니다. 출력된 숫자와 함께 근거가 확인된 항목과 미확인 항목을 살펴봅니다.

In [24]:
# 실습 : full 에 "그 수준과 그 아래 수준의 항목이 모두 통과한 가장 높은 수준"을 넣으세요

def judge(results):
    full = 0
    for level in (1, 2, 3, 4):
        ### START CODE HERE ### (그 수준까지 모두 통과했으면 full 을 올리고 아니면 멈추기 4줄)
        # 1) 수준이 level 이하인 항목이 모두 "통과"이면 full 을 level 로 올리고, 하나라도 아니면 반복을 멈춥니다
        if all(r["state"] == "통과" for r in results if r["level"] <= level):
            full = level
        else:
            break
        ### END CODE HERE ###
    if full is None:
        return None
    mixed = [f"{r['no']}번(수준 {r['level']})" for r in results if r["level"] > full and r["state"] == "통과"]
    passed = {r["no"] for r in results if r["state"] == "통과"}
    google = 2 if {11, 12} <= passed else 1 if 11 in passed else 0
    return {"마이크로소프트": full, "섞인 특징": mixed, "구글": google}

<details class="lms-answer">
<summary>풀이 보기</summary>

<strong>빈칸 정답 코드 (`### START CODE HERE ###` 영역)</strong>

```python
if all(r["state"] == "통과" for r in results if r["level"] <= level):
    full = level
else:
    break
```

구글 수준은 어림으로 계산합니다. 파이프라인(11번)이 실행되면 1이고 테스트를 거친 자동 배포(12번)까지 있으면 2로 칩니다.

</details>

In [25]:
verdict = judge(CHECK) if CHECK else None
if verdict is None:
    print("judge 가 비어 있거나 채점 결과가 없습니다. 실습 셀을 채우고 다시 실행하세요")
else:
    n_pass = sum(r["state"] == "통과" for r in CHECK)
    print(f"[측정 4] 통과 {n_pass}/{len(CHECK)}항목 / 판정 : {verdict}\n\n다음 할 일 후보 (수준이 낮은 항목부터) :")
    for r in sorted([r for r in CHECK if r["state"] != "통과"], key=lambda r: (r["level"], r["no"]))[:5]:
        print(f" - {r['no']}번 {r['item']} : {r['evidence']}")
    # 점검 결과도 트래킹 서버에 남깁니다(이론 노트 4.3절의 6번)
    mlflow.set_experiment("수준 점검")
    with mlflow.start_run(run_name=time.strftime("점검 %Y-%m-%d %H:%M")):
        mlflow.log_metrics({"passed": n_pass, "ms_level": verdict["마이크로소프트"], "google_level": verdict["구글"]})
        mlflow.log_artifact(str(WORK / "level_check.json"))
    mlflow.set_experiment(EXP)

[측정 4] 통과 4/13항목 / 판정 : {'마이크로소프트': 0, '섞인 특징': ['4번(수준 2)', '5번(수준 2)', '9번(수준 3)', '10번(수준 3)'], '구글': 0}

다음 할 일 후보 (수준이 낮은 항목부터) :
 - 1번 버전을 고정해 설치한다 : versions 필드가 빠진 실행 6/22
 - 2번 결과마다 조건을 적는다 : gpu·model 필드가 빠진 실행 9/22
 - 3번 질문 세트를 고정한다 : qset 필드가 빠진 실행 2/22
 - 6번 버전마다 카드가 있다 : 버전 3개, 카드가 빈 버전 ['3']
 - 7번 다른 사람이 처음부터 다시 만든다 : 증거 파일 repro.json 없음 (다른 사람의 계정에서 다시 만든 기록)


**눈여겨볼 곳** : mock 파일로 실행했다면 수준 1의 1번이 불합격이라 마이크로소프트 수준은 0으로 나옵니다. 그런데 5번, 9번 같은 높은 수준의 항목은 이미 통과해 "섞인 특징"으로 출력됩니다.

## Step 8. 선택 심화 트랙

본 실습이 끝나고 시간이 남으면 트랙 1을 실행합니다.

| 자리 | 주제 | 성격 | 난이도 |
|---|---|---|---|
| 트랙 1 | 게이트를 거친 운영 별칭 이동 | 확장 | 중 |
| 부록 실습 B | 트래킹 서버 백업과 복구 | 운영 | 중 |
| 부록 실습 E | 인증을 켠 트래킹 서버 | 보안 | 상 |

#### 트랙 1. 평가 기준을 통과한 버전으로 운영 별칭 바꾸기
기준선과 후보의 실행 기록은 오늘 생성한 예제를 사용합니다.

이론 노트 3.3절의 게이트를 별칭 옮기기에 붙입니다. 후보 버전이 게이트를 통과할 때만 "운영"을 옮기고 통과하지 못하면 까닭을 버전의 태그에 남깁니다.

In [26]:
def measured(mv):
    model_id = mv.source.split("/")[-1] if mv.source.startswith("models:/") else None
    run_id = mlflow.get_logged_model(model_id).source_run_id if model_id else None
    return client.get_run(run_id) if run_id else None


# 실습 : 게이트를 통과하지 못한 까닭을 reasons 에 담으세요. 비어 있으면 통과입니다

def gate(candidate):
    cur = client.get_model_version_by_alias(NAME, "운영")
    a, b = measured(cur), measured(client.get_model_version(NAME, candidate))
    reasons = []
    if b is None:
        return ["후보 버전에 측정한 실행이 없음"]
    if a is None or a.data.tags.get("qset") != b.data.tags.get("qset"):
        reasons.append("운영 버전과 질문 세트가 다름")
    if a is not None and b.data.metrics.get("correct", -1) < a.data.metrics.get("correct", 0):
        reasons.append(f"정답 {b.data.metrics.get('correct')} < 운영 {a.data.metrics.get('correct')}")
    ### START CODE HERE ### (flip 이 0 이 아니면 까닭 담기 2줄. flip 이 기록되지 않았으면 불합격으로 칩니다)
    # 1) flips_lost 가 없으면 1 로 읽어(기본값이 불합격) 0 이 아닐 때 까닭을 담습니다
    if b.data.metrics.get("flips_lost", 1) != 0:
        reasons.append(f"flip {b.data.metrics.get('flips_lost', '기록 없음')}")
    ### END CODE HERE ###
    return reasons

<details class="lms-answer">
<summary>풀이 보기</summary>

<strong>빈칸 정답 코드 (`### START CODE HERE ###` 영역)</strong>

```python
if b.data.metrics.get("flips_lost", 1) != 0:
    reasons.append(f"flip {b.data.metrics.get('flips_lost', '기록 없음')}")
```

측정한 실행이 없으면 더 볼 것 없이 불합격입니다. flip이 기록되지 않은 버전도 0으로 치지 않고 불합격으로 봅니다(판정 함수는 불합격이 기본값).

</details>

In [27]:
def promote(candidate):
    reasons = gate(candidate)
    if reasons is None:
        return "gate 가 비어 있습니다. 실습 셀을 채우세요"
    if reasons:
        client.set_model_version_tag(NAME, candidate, "gate", "; ".join(reasons))
        return f"버전 {candidate} 불합격 : {'; '.join(reasons)}"
    cur = aliases_of()["운영"]
    client.set_registered_model_alias(NAME, "운영", candidate)
    client.set_registered_model_alias(NAME, "직전", cur)
    return f"버전 {candidate} 통과 : 운영을 {cur}에서 {candidate}로 옮김"


if "운영" not in aliases_of():
    print("운영 별칭이 없어 게이트를 테스트할 수 없습니다. Step 6의 등록 셀을 먼저 실행하세요")
else:
    # 후보 1 : 답 길이 상한만 바꾸고 아직 측정하지 않은 버전
    base_params = mlflow.get_logged_model(client.get_model_version_by_alias(NAME, "운영").source.split("/")[-1]).params
    lm = mlflow.create_external_model(name="rag-service", params={**base_params, "max_tokens": "80"},
                                      model_type="rag-service")
    print(promote(mlflow.register_model(f"models:/{lm.model_id}", NAME,
                                        tags={"label": "답 길이 80 (아직 측정 안 함)"}).version))
    # 후보 2 : 해커톤 기준선 버전
    if "기준선" in VERSIONS:
        result = promote(VERSIONS["기준선"])
        print(result)
        if "통과" in result:
            print("연습이 끝났으니 롤백합니다 :", rollback())
    print("별칭 :", aliases_of())

Registered model '사내규정-RAG' already exists. Creating a new version of this model...
2026/11/24 04:35:39 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: 사내규정-RAG, version 4
Created version '4' of model '사내규정-RAG'.


버전 4 불합격 : 후보 버전에 측정한 실행이 없음
버전 1 불합격 : 정답 20.0 < 운영 22.0
별칭 : {'운영': '2', '직전': '1'}


평가 실행이 연결되지 않은 버전은 지표를 비교하기 전에 차단합니다. 필수 근거가 없을 때 통과시키지 않아야, 검증하지 않은 후보가 자동으로 운영 대상으로 선택되는 일을 막을 수 있습니다.

## Step 9. 공유와 정리

본 실습은 Step 6 뒤 이 단계로 이어집니다. 이 런타임에서 기동한 서버를 종료하고 입력·DB·아티팩트·변경 기록을 ZIP으로 보관합니다. 부록을 계속하면 `ensure_mlflow()`로 같은 기록 저장소의 서버를 다시 기동합니다.

In [28]:
for name, proc in SERVERS.items():
    stop_server(proc)
    print(f"{name} 서버 종료")
for key in ["MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"]:
    os.environ.pop(key, None)
if SERVERS.get("main") is not None:
    os.environ.pop("MLFLOW_TRACKING_URI", None)  # 재실행 때 종료한 로컬 서버를 사내 서버로 오인하지 않게 합니다.
print("남긴 파일:", sorted(p.name for p in WORK.iterdir()))

import shutil
bundle = shutil.make_archive(str(WORK.parent / f"d66_학습기록_{INPUT_MODE}"), "zip", root_dir=WORK)
print("학습 기록 묶음:", bundle)

main 서버 종료
남긴 파일: ['artifacts', 'backup', 'evidence', 'inventory.json', 'ledger.jsonl', 'level_check.json', 'mlflow.db', 'mlflow.log', 'standin']
학습 기록 묶음: /content/d66/d66_학습기록_sample.zip


출력된 `d66_학습기록_sample.zip`을 코랩 파일 창에서 내려받습니다. 작업 폴더를 바꿨다면 출력된 실제 경로를 사용합니다. ZIP에는 이번에 생성한 입력과 MLflow의 `mlflow.db`, `artifacts`, `inventory.json`, 별칭 변경 기록이 포함됩니다. DB와 아티팩트를 함께 보관해야 지표에서 원본 파일까지 다시 찾을 수 있습니다.

### 결과 확인

Step 5에서 비교가 보류된 실행의 누락 조건과 Step 6에서 출력한 기준선·고른 설정의 버전 번호를 기록합니다. 처음의 `운영` 버전과 롤백 직후의 `운영` 버전을 함께 남깁니다.

### 5분 마무리 확인

다른 팀원에게 실행 두 개를 비교할 수 있는지 설명합니다. 조건 하나와 그 근거가 된 원본 파일 또는 태그를 보여 줍니다. 이어서 롤백 전후 `운영`이 가리킨 버전을 말하고, 실제 사용자 요청이 복구됐는지 확인하려면 무엇을 검사할지 한 가지 답합니다.

| 자기 점검 | 남길 근거 |
|---|---|
| [목표 1] 평가 결과의 원본을 찾을 수 있는가? | 실행 ID와 연결된 결과 파일 |
| [목표 2] 비교하거나 보류한 이유를 설명할 수 있는가? | 질문 세트·모델·GPU 태그와 실측·대역 표시 |
| [목표 3] 되돌릴 버전과 실제 서비스 검사 방법을 설명할 수 있는가? | 롤백 전후 운영 별칭과 복구 확인에 쓸 정상 요청 |

컨테이너 심화·쿠버네티스 노트(D67)에서는 같은 서비스를 다른 환경에서도 실행하도록 필요한 프로그램·라이브러리·설정을 묶습니다.

## 부록 실습 A. 답마다 버전 기록

교재 : 7.4.4 LLM 추론

본 실습을 마친 뒤 시간이 남거나 더 깊이 보고 싶을 때 하는 실습입니다. 본 실습 Step 6까지 실행한 상태에서 시작합니다. 선택 Step 7의 채점기는 선행 조건이 아닙니다. Step 9에서 서버를 종료했다면 부록마다 첫 셀의 `ensure_mlflow()`가 트래킹 서버를 다시 기동합니다. 서비스가 "운영" 별칭을 읽어 버전을 정하고 답할 때마다 요청 기록에 버전 번호를 적게 만듭니다. 생성은 mock입니다. 서비스는 별칭을 요청마다 묻지 않고 정해진 주기(`ttl`)마다 다시 읽습니다.

In [29]:
ensure_mlflow()

# 본 실습 Step 6에서 등록한 버전 하나의 카드를 출력해 봅니다
if VERSIONS:
    mv = client.get_model_version(NAME, VERSIONS.get("고른 설정", next(iter(VERSIONS.values()))))
    print(mv.description)

# 함정 : search_model_versions 의 결과에는 별칭 필드가 비어 있습니다. 별칭은 get_registered_model 로 읽습니다
print("\nsearch_model_versions 의 별칭 필드 :", [v.aliases for v in client.search_model_versions(f"name='{NAME}'")])

트래킹 서버를 다시 기동했습니다 : http://127.0.0.1:8662 / 기동까지 6.1 초
# 사내규정-RAG : 고른 설정
- 원본 : 큰 모델 Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4 / 작은 모델 Qwen/Qwen2.5-1.5B-Instruct / 임베딩 intfloat/multilingual-e5-small
- 서비스 설정 : 후보 20, 넣는 청크 3, 답 길이 상한 120, 순서 good-id, 프롬프트 버전 D66-예제-v1
- 잰 조건 : GPU 빠짐 / 질문 세트 be85881f3606 / 모드 대역 / 버전 빠짐
- 결과 : 정답 22/25, 적중 25, 전체 p50 3.1초, p95 4.34초
- 잰 실행 : c50ad0288fff4ab2890b781fa84158a3

search_model_versions 의 별칭 필드 : [[], [], [], []]


In [30]:
LEDGER = WORK / "ledger.jsonl"


class Service:
    """'운영' 별칭이 가리키는 서비스 버전으로 답하는 mock 서버입니다"""

    def __init__(self, name, alias="운영", ttl=2.0, ledger=LEDGER):
        self.name, self.alias, self.ttl, self.ledger = name, alias, ttl, ledger
        self.cached, self.read_at = None, 0.0

    def current(self):
        if self.cached is None or time.perf_counter() - self.read_at > self.ttl:
            mv = client.get_model_version_by_alias(self.name, self.alias)
            model_id = mv.source.split("/")[-1] if mv.source.startswith("models:/") else None
            params = mlflow.get_logged_model(model_id).params if model_id else {}
            self.cached, self.read_at = (mv.version, params), time.perf_counter()
        return self.cached

    def answer(self, qid):
        version, params = self.current()
        text = f"(mock 답) {qid} : 청크 {params.get('keep')}개, 답 길이 상한 {params.get('max_tokens')}"
        rec = {"ts": round(time.time(), 3), "qid": qid, "version": version, "prompt_version": params.get("prompt_version")}
        with open(self.ledger, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        return version, text


LEDGER.unlink(missing_ok=True)
svc = Service(NAME, ttl=float(os.environ.get("D66_ALIAS_TTL", "2")))
if "운영" not in aliases_of():
    print("운영 별칭이 없어 답하지 않습니다. 본 실습 Step 6의 등록 셀을 먼저 실행하세요")
else:
    for q in ["Q01", "Q02", "Q03"]:
        print(svc.answer(q))

('2', '(mock 답) Q01 : 청크 3개, 답 길이 상한 120')
('2', '(mock 답) Q02 : 청크 3개, 답 길이 상한 120')
('2', '(mock 답) Q03 : 청크 3개, 답 길이 상한 120')


In [31]:
moved = rollback()
if moved is None:
    print("롤백하지 못했습니다. rollback 이 비어 있거나 별칭이 없습니다. 본 실습 Step 6의 실습 셀을 채우고 이 셀을 다시 실행하세요")
else:
    print(f"별칭을 옮겼습니다 : {moved}")
    t0, served = time.perf_counter(), None
    while time.perf_counter() - t0 < svc.ttl + 5:
        served, _ = svc.answer("Q04")
        if served == moved["운영"]:
            break
        time.sleep(0.2)
    switch_s = round(time.perf_counter() - t0, 2)
    print(f"[부록 A] 별칭을 옮긴 뒤 옛 버전({moved['운영']})으로 답하기까지 {switch_s}초 (별칭을 다시 읽는 주기 {svc.ttl}초)")
    back = rollback()                                        # 연습이 끝났으니 다시 앞으로 옮깁니다
    print("다시 앞으로 :", back)
    _ = (EVID / "rollback.json").write_text(json.dumps(
        {"date": time.strftime("%Y-%m-%d %H:%M"), "moved": moved, "switch_s": switch_s, "back": back},
        ensure_ascii=False, indent=1), encoding="utf-8")

lines = [json.loads(l) for l in LEDGER.read_text(encoding="utf-8").splitlines() if l.strip()] if LEDGER.exists() else []
print("\n요청 기록 :", len(lines), "건 / 버전별 :", dict(Counter(x["version"] for x in lines)))

별칭을 옮겼습니다 : {'운영': '1', '직전': '2', 'ms': 75.5}
[부록 A] 별칭을 옮긴 뒤 옛 버전(1)으로 답하기까지 1.65초 (별칭을 다시 읽는 주기 2.0초)
다시 앞으로 : {'운영': '2', '직전': '1', 'ms': 102.2}

요청 기록 : 12 건 / 버전별 : {'2': 11, '1': 1}


**출력에서 확인**: 별칭을 바꾼 시각과 응답의 버전이 바뀐 시각을 비교합니다. 모의 서버가 별칭을 정해진 주기마다 다시 읽으므로 두 시각은 다를 수 있습니다. 조회 주기를 줄이면 변경을 더 빨리 확인할 수 있지만 레지스트리로 보내는 요청은 늘어납니다.

요청 기록이 생겼으니 Step 7의 채점 셀을 다시 실행하면 10번이 통과합니다.

## 부록 실습 B. 트래킹 서버 백업과 복구

교재 : 9.8.1 LLM 애플리케이션의 백업 및 장애 대비 프로세스, 9.8.3 정기적인 복구 테스트의 중요성

이론 노트 4.4절은 트래킹 서버도 운영 대상이라고 했습니다. 실행 중인 서버의 기록 파일과 산출물 폴더를 백업하고 그 백업으로 서버를 하나 더 기동해 내용이 같은지 봅니다. 첫 셀의 `snapshot`은 이번 백업과 복구 비교에 사용합니다.

In [32]:
ensure_mlflow()


def snapshot(uri, exp_name=EXP, name=NAME):
    """기록 저장소 하나의 요약 : 실행 수, 버전 수, 운영과 직전 별칭"""
    c = MlflowClient(uri)
    e = c.get_experiment_by_name(exp_name)
    n_runs = len(c.search_runs([e.experiment_id], max_results=5000)) if e else 0
    try:
        n_versions = len(c.search_model_versions(f"name='{name}'"))
        aliases = c.get_registered_model(name).aliases
    except Exception:
        n_versions, aliases = 0, {}
    return {"실행": n_runs, "버전": n_versions, "운영": aliases.get("운영"), "직전": aliases.get("직전")}


print(snapshot(TRACKING_URI))

{'실행': 22, '버전': 4, '운영': '2', '직전': '1'}


In [33]:
ensure_mlflow()

import shutil
import sqlite3

BK = WORK / "backup"

# 실습 : 실행 중인 서버의 SQLite 파일을 안전하게 복사하세요. 끝나면 dest 를 반환합니다

def backup_sqlite(db, dest):
    src, dst = sqlite3.connect(db), sqlite3.connect(dest)
    with dst:
        ### START CODE HERE ### (원본 연결의 backup() 으로 사본에 복사하기 1줄)
        # 1) 쓰는 중인 파일을 그냥 복사하면 내용이 어긋날 수 있으므로 원본 연결(src)의 backup() 으로 dst 에 복사합니다
        src.backup(dst)
        ### END CODE HERE ###
    src.close()
    dst.close()
    done = dest
    return done

<details class="lms-answer">
<summary>풀이 보기</summary>

<strong>빈칸 정답 코드 (`### START CODE HERE ###` 영역)</strong>

```python
src.backup(dst)
```

`backup()`은 다른 프로세스가 쓰고 있는 데이터베이스도 한 시점의 모습으로 복사합니다.

</details>

In [34]:
if SERVERS.get("main") is None:
    print("이 런타임에서 기동한 트래킹 서버가 없어 건너뜁니다. 🖥 사내 트래킹 서버의 백업은 그 서버 담당자와 함께 합니다")
else:
    BK.mkdir(exist_ok=True)
    t0 = time.perf_counter()
    done = backup_sqlite(WORK / "mlflow.db", BK / "mlflow.db")
    if done is None:
        print("backup_sqlite 가 비어 있습니다. 실습 셀을 채우고 이 셀을 다시 실행하세요")
    else:
        shutil.copytree(WORK / "artifacts", BK / "artifacts", dirs_exist_ok=True)
        port2 = pick_port(PORT + 1, taken={PORT})
        SERVERS["restore"] = start_server(BK / "mlflow.db", BK / "artifacts", port2)
        uri2 = f"http://127.0.0.1:{port2}"
        if wait_health(uri2) is None:
            print("복구 서버가 기동되지 않았습니다 :", BK / "mlflow.log")
        else:
            a, b = snapshot(TRACKING_URI), snapshot(uri2)
            print(f"원래 : {a}\n복구 : {b}")
            # 산출물까지 돌아왔는지 실행 하나의 원본 파일을 복구 서버에서 다운로드해 봅니다
            r0 = MlflowClient(uri2).search_runs([MlflowClient(uri2).get_experiment_by_name(EXP).experiment_id],
                                                max_results=1)[0]
            got = mlflow.artifacts.download_artifacts(run_id=r0.info.run_id, artifact_path="raw",
                                                      tracking_uri=uri2, dst_path=str(BK / "check"))
            print("복구 서버에서 받은 원본 :", sorted(p.name for p in Path(got).iterdir()))
            print(f"[부록 B] 백업부터 복구 서버 확인까지 {time.perf_counter() - t0:.1f}초, 요약이 같음 : {a == b}")

원래 : {'실행': 22, '버전': 4, '운영': '2', '직전': '1'}
복구 : {'실행': 22, '버전': 4, '운영': '2', '직전': '1'}


복구 서버에서 받은 원본 : ['d60_results.json']
[부록 B] 백업부터 복구 서버 확인까지 6.8초, 요약이 같음 : True


요약이 같고 원본 파일까지 다운로드되면 복구가 됐다고 판정합니다.

## 부록 실습 E. 인증을 켠 트래킹 서버

교재 : 6.4 자격 증명 관리

이론 노트 4.4절에서는 기록을 읽는 권한과 운영 버전을 바꾸는 권한을 구분했습니다. 이번에는 그 차이를 인증을 켠 서버에서 확인합니다. MLflow의 기본 인증(basic-auth)을 켠 서버를 하나 더 기동해 로그인하지 않은 요청과 읽기 권한만 있는 사용자가 무엇을 거절당하는지 봅니다.

In [35]:
%pip install -q "mlflow[auth]==3.16.1"

In [36]:
ensure_mlflow()  # Step 9에서 종료했어도 오늘 만든 기록 서버를 다시 기동합니다.

import secrets
import shutil

AUTH = WORK / "auth"
old = SERVERS.get("auth")                      # 셀을 다시 실행했다면 앞서 기동한 연습 서버를 먼저 종료합니다
if old is not None and old.poll() is None:
    stop_server(old)
shutil.rmtree(AUTH, ignore_errors=True)        # 연습용 서버라 사용자 데이터베이스(basic_auth.db)까지 매번 새로 만듭니다
AUTH.mkdir(parents=True)
ADMIN_PW = os.environ.get("D66_ADMIN_PASSWORD") or secrets.token_urlsafe(16)    # 비밀번호를 코드에 적지 않습니다
port3 = pick_port(int(os.environ.get("D66_AUTH_PORT", "8665")), taken=set(range(8660, 8665)))
SERVERS["auth"] = start_server(AUTH / "store.db", AUTH / "artifacts", port3,
                               extra_env={"MLFLOW_AUTH_ADMIN_PASSWORD": ADMIN_PW,
                                          "MLFLOW_FLASK_SERVER_SECRET_KEY": secrets.token_hex(16)},
                               extra_args=["--app-name", "basic-auth"], cwd=AUTH)
AUTH_URI = f"http://127.0.0.1:{port3}"
print("인증 서버 :", AUTH_URI, "/ 기동까지", wait_health(AUTH_URI, limit_s=120), "초 (None 이면 기동되지 않음 :", AUTH / "store.log", ")")
os.environ["MLFLOW_HTTP_REQUEST_MAX_RETRIES"] = "0"            # 거절을 재시도하며 기다리지 않습니다
VIEWER_PW = secrets.token_urlsafe(12)


def login(user, pw):
    """이 노트북의 MLflow 요청에 붙일 사용자 이름과 비밀번호"""
    if user is None:
        os.environ.pop("MLFLOW_TRACKING_USERNAME", None)
        os.environ.pop("MLFLOW_TRACKING_PASSWORD", None)
    else:
        os.environ["MLFLOW_TRACKING_USERNAME"], os.environ["MLFLOW_TRACKING_PASSWORD"] = user, pw


def first_line(e):
    return f"{type(e).__name__}: {str(e).splitlines()[0][:110]}"


login(None, None)
try:
    MlflowClient(AUTH_URI).search_experiments()
    print("로그인 없이 : 통과 (인증이 꺼져 있습니다)")
except Exception as e:
    print("로그인 없이 :", first_line(e))

login("admin", ADMIN_PW)
auth_client = MlflowClient(AUTH_URI)
mlflow.set_tracking_uri(AUTH_URI)
try:
    mlflow.set_experiment("인증 서버")
    lm = mlflow.create_external_model(name="rag-service", params={"keep": "3"}, model_type="rag-service")
    v1 = mlflow.register_model(f"models:/{lm.model_id}", NAME).version
    auth_client.set_registered_model_alias(NAME, "운영", v1)
    from mlflow.server import get_app_client
    get_app_client("basic-auth", tracking_uri=AUTH_URI).create_user(username="viewer", password=VIEWER_PW)
    print("관리자 : 버전 등록, 운영 별칭, 읽기 사용자 viewer 만들기 완료")
except Exception as e:
    print("관리자 작업 실패 :", first_line(e))
finally:
    login(None, None)
    mlflow.set_tracking_uri(TRACKING_URI)
    mlflow.set_experiment(EXP)

인증 서버 : http://127.0.0.1:8665 / 기동까지 9.8 초 (None 이면 기동되지 않음 : /content/d66/sample/auth/store.log )
로그인 없이 : MlflowException: API request to endpoint /api/2.0/mlflow/experiments/search failed with error code 401 != 200. Response body: '


2026/11/24 04:36:15 INFO mlflow.tracking.fluent: Experiment with name '인증 서버' does not exist. Creating a new experiment.
Successfully registered model '사내규정-RAG'.
2026/11/24 04:36:18 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: 사내규정-RAG, version 1
Created version '1' of model '사내규정-RAG'.
/usr/local/lib/python3.13/dist-packages/mlflow/server/fastapi_app.py:23: StarletteDeprecationWarning: starlette.middleware.wsgi is deprecated and will be removed in a future release. Please refer to https://github.com/abersheeran/a2wsgi as a replacement.
  from starlette.middleware.wsgi import WSGIResponder, build_environ


관리자 : 버전 등록, 운영 별칭, 읽기 사용자 viewer 만들기 완료


In [37]:
# 실습 : viewer 로 로그인해 운영 별칭을 읽고, 별칭을 옮기려 해 보세요

login("viewer", VIEWER_PW)
viewer = MlflowClient(AUTH_URI)
result = {}
try:
    ### START CODE HERE ### (viewer 로 운영 별칭을 읽고 같은 버전으로 옮겨 보기 2줄)
    # 1) 운영 별칭이 가리키는 버전 번호를 읽습니다
    result["읽기"] = viewer.get_model_version_by_alias(NAME, "운영").version
    # 2) 같은 버전으로 운영 별칭을 옮겨 봅니다. 읽기 권한뿐이면 여기서 거절됩니다
    viewer.set_registered_model_alias(NAME, "운영", result["읽기"])
    ### END CODE HERE ###
    result["옮기기"] = "됨 (권한 설정을 다시 보세요)"
except Exception as e:
    result["옮기기" if "읽기" in result else "읽기"] = first_line(e)
login(None, None)
print(result if result is not None else "result 가 비어 있습니다. 실습 셀을 채우세요")

{'읽기': '1', '옮기기': 'MlflowException: API request to endpoint /api/2.0/mlflow/registered-models/alias failed with error code 403 != 200. Response bo'}


<details class="lms-answer">
<summary>풀이 보기</summary>

<strong>빈칸 정답 코드 (`### START CODE HERE ###` 영역)</strong>

```python
result["읽기"] = viewer.get_model_version_by_alias(NAME, "운영").version
viewer.set_registered_model_alias(NAME, "운영", result["읽기"])
```

로그인하지 않은 요청은 401, 읽기 권한만 있는 viewer의 별칭 옮기기는 403으로 거절됩니다.

</details>

**로컬 작업** : 사내에서는 트래킹 서버의 인증을 회사 계정 체계(SSO 등)에 연결하고 운영 별칭을 옮길 수 있는 사람을 정해 둡니다.

끝나면 Step 9의 종료 셀을 다시 실행해 연습 서버까지 종료합니다.